In [1]:
import os
import shutil
import collections
import random
import imageio

import numpy as np
import pandas as pd
import nibabel as nib

import imageio.v2 as imageio

from skimage.transform import resize 
from skimage import morphology
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path



### Define Variables and Pathnames
- t1n = T1-weighted pre-contrast
- t1c = T1-weighted post-contrast
- t2w = T2-weighted
- t2f = FLAIR
- seg = segmentation mask

In [ ]:

data_path = '/path/to/nifti/patient/files'
output_path = '/desired/output/path'

img_size = [256, 256]    # change to [512, 512] for StyleGAN2-ADA

modalities = ['t1n','t1c','t2w','t2f','seg']



## Transform to PNG and normalize 
- create PNG from nifti files per patient
- normalize to [0,255]
- resize images to (256 x 256)

In [ ]:


# ========================
# Normalize MRIs to [0,255]
# ========================
def normalize_mri(img):
    img = img.astype(np.float32)
    img = img - img.min()
    if img.max() != 0:
        img = img / img.max()
    img *= 255.0
    return img.astype(np.uint8)


patients = sorted(os.listdir(data_path))

for patient in patients:
    patient_path = os.path.join(data_path, patient)
    if not os.path.isdir(patient_path):
        continue

    print(f'preprocessing patient: {patient}')

    # create output folders for each modality

    patient_out = {}
    for mod in modalities:
        mod_out = os.path.join(output_path, patient, mod)
        os.makedirs(mod_out, exist_ok=True)
        patient_out[mod] = mod_out
        

    # load all modality NIfTIs

    images = {}
    for mod in modalities:
        for f in os.listdir(patient_path):
            if f.endswith(".nii.gz") and mod in f:
                img_path = os.path.join(patient_path, f)
                img = nib.load(img_path).get_fdata()
                images[mod] = img
                print(f'    loaded {mod}: {f}, shape: {img_size}')

    # make sure all volumes have the same number of slices
    n_slices = images['t1n'].shape[2]


    # loop over slices
    for z in range(n_slices):
        for mod in modalities:
            slice_img = images[mod][:, :, z]
            if mod != 'seg':
                slice_img = normalize_mri(slice_img)
            else:
                slice_img = slice_img.astype(np.uint8)

            # resize to target image size

            slice_img_resized = resize(slice_img, img_size, 
                                       order=0 if mod=='seg' else 1,
                                       preserve_range=True, anti_aliasing=True)
            slice_img_resized = slice_img_resized.astype(np.uint8)

            # save PNG
            outfile = os.path.join(patient_out[mod], f'{str(z).zfill(3)}.png')
            imageio.imwrite(outfile, slice_img_resized, comppress_level=0)



## Split patients into train, test, and validation
- 900 train, 100 validation, 251 test patients
- save split as txt files to reference for model data splitting


In [ ]:


source_root = output_path
split_data_output = '/desired/split/data/output/path'

# find patient IDs
all_patients = sorted([
    p for p in os.listdir(source_root) if os.path.isdir(os.path.join(source_root, p))
])

total = len(all_patients)
assert total = 1251, f"Expected 1251 patients, found {total}"

# Shuffle and split
random.seed(42)          # remove or change seed for a different draw
random.shuffle(all_patients)

train_ids = all_patients[:900]
validation_ids = all_patients[900:1000]
test_ids = all_patients[1000:1251]

# Create txt files for patient splits
os.makedirs(split_data_output, exist_ok=True)

split_files = {
    "train": os.path.join(split_data_output, "train_patients.txt"),
    "validation": os.path.join(split_data_output, "validation_patients.txt"),
    "test": os.path.join(split_data_output, "test_patients.txt"),
}

split_ids = {
    "train": train_ids,
    "validation": validation_ids,
    "test": test_ids,
}

for split, patient_file in split_files.items():
    with open(patient_file, "w") as f:
        f.write("\n".join(split_ids[split]) + "\n")
    print(f" Wrote {len(split_ids[split])}, IDs: {patient_file}")


In [ ]:
# Copy patient folders into split folders

copied, missing = 0, 0
missing_patients = []

for split, patient_file in split_files.items():
    with open(patient_file, "r") as f:
        patient_ids = [line.strip() for line in f if line.strip()]

    split_dest = os.path.join(split_data_output, split)
    os.makedirs(split_dest, exist_ok=True)

    for patient_id in patient_ids:
        src_patient = os.path.join(source_root, patient_id)
        dst_patient = os.path.join(split_dest, patient_id)

        if not os.path.isdir(src_patient):
            print(f"[WARN] Source folder not found: {src_patient}")
            missing += 1
            missing_patients.append(patient_id)
            continue

        shutil.copytree(src_patient, dst_patient, dirs_exist_ok=True)
        copied += 1
        print(f" {split}/{patient_id}")

print(f"Copied: {copied} patients. Missing: {missing} patients")
if missing_patients:
    print("\nMissing patients:")
    for p in missing_patients:
        print(f"{p}")

## Prune data
- 15 axial slices per patient
- less than 5% brain matter slices are excluded

In [ ]:

split_paths = {
    "train": '/path/to/train/patients',
    "validation": '/path/to/validation/patients',
    "test": '/path/to/test/patients',
}

modalities = ['t1n','t1c','t2w','t2f','seg']

K = 15
fg_thresh = 0.05  # 5% non-background
bg_val = 0        # adjust if background is not exactly zero

def foreground_fraction(img):
    return np.mean(img > bg_val)


In [3]:
for split, split_path in split_paths.items():     
    
    patients = sorted([
        p for p in os.listdir(split_path)
        if os.path.isdir(os.path.join(split_path, p))
    ])

    for patient in patients:
        patient_path = os.path.join(split_path, patient)   # ← use split_path, not data_path

        print(f"Processing {patient}")

        # Use T1n slices as reference
        t1n_path = os.path.join(patient_path, "t1n")
        slice_files = sorted(f for f in os.listdir(t1n_path) if f.endswith(".png"))

        usable_indices = []
        for f in slice_files:
            img = imageio.imread(os.path.join(t1n_path, f))
            if foreground_fraction(img) >= fg_thresh:
                usable_indices.append(int(f.replace(".png", "")))

        if len(usable_indices) < K:
            print(f"Skipping {patient} (only {len(usable_indices)} usable slices)")
            continue

        # Select K evenly spaced slices from *within* the usable list  ← fix
        selected_positions = np.linspace(0, len(usable_indices) - 1, K, dtype=int)
        selected = [usable_indices[i] for i in selected_positions]

        # Copy slices for all modalities
        for mod in modalities:                                          # ← was MRI_modalities
            mod_in  = os.path.join(patient_path, mod)
            mod_out = os.path.join(split_path, patient, mod)           # ← output within split
            os.makedirs(mod_out, exist_ok=True)

            for z in selected:
                fname = f"{str(z).zfill(3)}.png"
                src = os.path.join(mod_in, fname)
                dst = os.path.join(mod_out, fname)
                if os.path.exists(src):
                    shutil.copy2(src, dst)
                else:
                    print(f"Missing: {src}")

# Prune data (SynDiff only)
- prune 100 axial slices (instead of 15) per patient
- less than 5% brain matter excluded

### Prune on the original PNG data folder

In [ ]:

original_data = 'your/pre-pruned/data'
pruned_path_syndiff = 'your/syndiff/pruned/data/path'

K_new = 100

patients = sorted(os.listdir(original_data))

for patient in patients:
    patient_path = os.path.join(original_data, patient)
    print(f"Found {len(patients)} entries in original_data")
    print(patients[:5])
    if not os.path.isdir(patient_path):
        continue

    print(f"Processing {patient}")

    # ---- load T1n slices as reference ----
    t1n_path = os.path.join(patient_path, "t1n")
    slice_files = sorted(f for f in os.listdir(t1n_path) if f.endswith(".png"))

    usable_indices = []
    for f in slice_files:
        img = imageio.imread(os.path.join(t1n_path, f))
        if foreground_fraction(img) > fg_thresh:
            usable_indices.append(int(f.replace(".png", "")))

    if len(usable_indices) < K_new:
        print(f"Skipping {patient} (only {len(usable_indices)} usable slices)")
        continue

    # ---- evenly spaced selection ----
    selected = np.linspace(
        usable_indices[0],
        usable_indices[-1],
        K_new,
        dtype=int
    ).tolist()

    # ---- copy slices for BOTH modalities ----
    for mod in modalities:
        mod_in = os.path.join(patient_path, mod)
        mod_out = os.path.join(output_path, patient, mod)
        os.makedirs(mod_out, exist_ok=True)

        for z in selected:
            fname = f"{str(z).zfill(3)}.png"
            src = os.path.join(mod_in, fname)
            dst = os.path.join(mod_out, fname)
            if os.path.exists(src):
                shutil.copy2(src, dst)
            else:
                print(f"Missing: {src}")

### Now do the split (using patient ID txt files)

In [ ]:

source_root = pruned_path_syndiff
split_dest_root  = '/your/split/data/output/path'  

split_files = {
    "train": "./train_patients.txt",               
    "validation": "./validation_patients.txt",
    "test": "./test_patients.txt",
}

for split, patient_file in split_files.items():
    with open(patient_file, "r") as f:
        patient_ids = [line.strip() for line in f if line.strip()]

    split_dest = os.path.join(split_dest_root, split)  # ← consistent variable
    os.makedirs(split_dest, exist_ok=True)

    for patient_id in patient_ids:
        
        src_patient = os.path.join(source_root, patient_id)
        dst_patient = os.path.join(split_dest, patient_id)

        shutil.copytree(src_patient, dst_patient, dirs_exist_ok=True)
       
print(f"\nDone!")
